# LoRA verstehen: ein Modell auf CVE-Triage trainieren

> 🛠️ **Workshop-Version:** Bearbeite die zwei Aufgaben. Die Lösungen kannst du direkt darunter aufklappen.

Am Ende kann ein kleines Sprachmodell aus einer CVE-Beschreibung ein festes
JSON-Objekt erzeugen:

```text
Beschreibung → Basismodell + LoRA-Adapter → Triage-JSON
```

**Danach kannst du:**

- entscheiden, wann Fine-Tuning sinnvoller ist als RAG,
- erklären, warum LoRA viel weniger Parameter trainiert,
- ein Trainingsbeispiel korrekt tokenisieren und maskieren,
- die Wirkung vor und nach dem Training messen.

**Zeit:** etwa 35 Minuten plus 3–7 Minuten Training. Es gibt nur zwei
Programmieraufgaben; der übrige Code ist bewusst vorgegeben.


## 0 · Setup

Führe die Zellen von oben nach unten aus. Beim ersten Start wird
`HuggingFaceTB/SmolLM2-135M` heruntergeladen; danach liegt es im lokalen Cache.
Das Training läuft mit CUDA, Apple Silicon oder CPU.


In [ ]:
try:
    import peft
except ImportError:
    %pip install -q peft transformers matplotlib

import json
import time

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

import helfer
from helfer import BLAU, GRAU

GERAET = helfer.waehle_geraet()
torch.manual_seed(1337)
print("Gerät:", GERAET)


In [ ]:
tokenizer, basis_modell = helfer.lade_modell(helfer.BASIS_MODELL, GERAET)
anzahl_parameter = sum(p.numel() for p in basis_modell.parameters())
print(f"Basismodell: {helfer.BASIS_MODELL}")
print(f"Parameter:   {anzahl_parameter:,}")


## 1 · Zuerst die richtige Methode wählen

Fine-Tuning ist nicht automatisch die richtige Lösung.

| Dein Problem | Passender Ansatz |
|---|---|
| Wissen ändert sich häufig oder braucht Quellen | **RAG**: Dokumente zur Laufzeit in den Prompt holen |
| Ausgabeformat oder Verhalten soll stabil sein | **Fine-Tuning**: Verhalten in den Gewichten verankern |
| Beides trifft zu | **Kombinieren**: Wissen per RAG, Verhalten per Fine-Tuning |

**Merksatz:** RAG liefert Wissen. Fine-Tuning übt Verhalten.

In diesem Notebook soll jede Antwort exakt dieselben JSON-Felder haben und
interne Triage-Regeln anwenden. Das ist ein gut abgegrenztes Verhalten — also
ein sinnvoller Fine-Tuning-Fall.

> **Kurz nachdenken:** Ein Runbook wurde gestern geändert. RAG oder
> Fine-Tuning? — **RAG**, denn die Information ist aktuell und veränderlich.


## 2 · Der gesamte Ablauf auf einen Blick

1. **Daten:** Beispiele aus Eingabe und gewünschter Antwort laden.
2. **Vorher messen:** Das unveränderte Modell auf ungesehenen Testfällen prüfen.
3. **Adapter einsetzen:** Das Basismodell einfrieren, nur kleine LoRA-Matrizen trainieren.
4. **Trainieren:** Fehler berechnen und nur den Adapter aktualisieren.
5. **Nachher messen:** Exakt denselben Test wiederholen.

So bleibt jederzeit klar, welchen Zweck die nächste Zelle erfüllt.


### Die Daten

Jedes Beispiel enthält eine englische CVE-Beschreibung und das gewünschte
Triage-JSON. Das Trainingsset dient zum Lernen; das Testset bleibt bis zur
Messung unangetastet.


In [ ]:
train = helfer.lade_jsonl("triage_train.jsonl")
test = helfer.lade_jsonl("triage_test.jsonl")

print(f"Training: {len(train)} Beispiele")
print(f"Test:     {len(test)} ungesehene Beispiele")
print()

beispiel = train[0]
print("EINGABE:", beispiel["beschreibung"])
print("ZIEL:   ", json.dumps(beispiel["ziel"]))


### Das Promptformat

Beim Training stehen Eingabe und Ziel direkt hintereinander. Der Trenner
`### TRIAGE` zeigt dem Modell: *Ab hier beginnt die Antwort.*

```text
### CVE
<Beschreibung>
### TRIAGE
<gewünschtes JSON>
```

Wichtig ist weniger die genaue Überschrift als ein **konsistentes Format** in
Training und Anwendung.


In [ ]:
TRENNER = "\n### TRIAGE\n"


def baue_prompt(beschreibung):
    return f"### CVE\n{beschreibung}{TRENNER}"


def baue_ziel(ziel):
    return json.dumps(ziel)


print(baue_prompt(beispiel["beschreibung"]) + baue_ziel(beispiel["ziel"]))


## 3 · Vor dem Training messen

Wir messen zwei Dinge auf dem Testset:

- **gültiges JSON:** Ist die Antwort überhaupt maschinenlesbar?
- **Feldtreue:** Stimmen `severity`, `component`, `action` und `owner`?

Die Auswertungsfunktion ist vorgegeben. Sie ist Messwerkzeug, nicht das
Lernziel dieses Workshops.


In [ ]:
FELDER = ["severity", "component", "action", "owner"]


def bewerte(modell, testfaelle):
    prompts = [baue_prompt(fall["beschreibung"]) for fall in testfaelle]
    antworten = helfer.erzeuge_antworten(modell, tokenizer, prompts)
    treffer = {feld: 0 for feld in FELDER}
    gueltig = 0

    for antwort, fall in zip(antworten, testfaelle):
        try:
            objekt = json.loads(antwort)
        except (json.JSONDecodeError, TypeError):
            continue
        if not isinstance(objekt, dict):
            continue
        gueltig += 1
        for feld in FELDER:
            treffer[feld] += objekt.get(feld) == fall["ziel"][feld]

    n = len(testfaelle)
    return {"json": gueltig / n, **{feld: treffer[feld] / n for feld in FELDER}}


vorher = bewerte(basis_modell, test)
for name, wert in vorher.items():
    print(f"{name:<12} {wert:>6.0%}")


In [ ]:
# Eine konkrete Antwort macht die Nullmessung anschaulich.
prompt_vorher = baue_prompt(test[0]["beschreibung"])
antwort_vorher = helfer.erzeuge_antworten(
    basis_modell, tokenizer, [prompt_vorher]
)[0]

print("MODELL:", antwort_vorher)
print("ZIEL:  ", json.dumps(test[0]["ziel"]))


## 4 · Was LoRA verändert

Beim vollständigen Fine-Tuning würden alle Gewichte des Modells verändert.
LoRA friert sie ein und ergänzt ausgewählte Schichten um eine kleine,
trainierbare Korrektur:

$$W_{neu} = W_{eingefroren} + B A$$

Die Matrizen `A` und `B` haben einen kleinen Rang `r`. Dadurch enthalten sie
viel weniger Werte als `W`.

**Bild im Kopf:** Das Basismodell ist ein dickes Lehrbuch. LoRA ersetzt nicht
das Buch, sondern legt wenige aufgabenspezifische Notizzettel hinein.

| Einstellung | Einfach erklärt |
|---|---|
| `r=4` | Größe bzw. Lernkapazität des Adapters |
| `lora_alpha=8` | Stärke der gelernten Korrektur |
| `target_modules` | Modellschichten, die Notizzettel erhalten |
| `lora_dropout=0.05` | kleine Regularisierung gegen Überanpassung |


### 🛠️ Aufgabe 1 — LoRA-Adapter anlegen

Ergänze die beiden Stellen `...`:

1. Erzeuge eine `LoraConfig` mit den angegebenen Werten.
2. Lege den Adapter mit `get_peft_model()` über das Basismodell.

Der Selbsttest zeigt anschließend, ob ausschließlich LoRA-Parameter
trainierbar sind.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = ...
modell = ...


In [ ]:
trainierbar = sum(p.numel() for p in modell.parameters() if p.requires_grad)
gesamt = sum(p.numel() for p in modell.parameters())
trainierbare_namen = [name for name, p in modell.named_parameters() if p.requires_grad]

assert trainierbar > 0, "Es wurden keine trainierbaren Parameter gefunden."
assert all("lora_" in name for name in trainierbare_namen),     "Außerhalb des LoRA-Adapters sind Parameter trainierbar."
assert trainierbar / gesamt < 0.01, "Der Adapter sollte unter 1 % bleiben."

print("✅ Adapter korrekt angelegt")
print(f"Trainierbar: {trainierbar:,} von {gesamt:,} Parametern")
print(f"Anteil:      {trainierbar / gesamt:.3%}")


<details>
<summary>💡 Lösung zu Aufgabe 1</summary>

```python
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
modell = get_peft_model(basis_modell, lora_config)
```

</details>


## 5 · Was das Modell im Training vorhersagen soll

Das Modell erhält Prompt und Ziel als eine Tokenfolge. Lernen soll es aber nur
die Antwort. Deshalb stehen in `labels` für alle Prompt-Tokens die Werte
`-100`; PyTorch ignoriert diese Positionen beim Loss.

```text
Tokens:  [-------- Prompt --------][------ Ziel ------][EOS]
Labels:  [-100, -100, ...,    -100][------ Ziel ------][EOS]
          wird nicht bewertet       wird gelernt       Stopp
```

Das EOS-Token ist wichtig: Es bringt dem Modell bei, nach dem JSON aufzuhören.


### 🛠️ Aufgabe 2 — Labels maskieren

Vervollständige `labels`:

- für jedes Prompt-Token einmal `-100`,
- danach die Token-IDs des Ziels.

Das ist die zentrale Datenvorbereitung beim überwachten Fine-Tuning eines
Causal Language Models.


In [ ]:
MAX_LAENGE = 256


def kodiere_beispiel(beispiel):
    prompt = baue_prompt(beispiel["beschreibung"])
    ziel = baue_ziel(beispiel["ziel"]) + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    ziel_ids = tokenizer(ziel, add_special_tokens=False)["input_ids"]

    input_ids = (prompt_ids + ziel_ids)[:MAX_LAENGE]
    labels = ...  # TODO: Prompt maskieren, Ziel beibehalten
    return {"input_ids": input_ids, "labels": labels}


In [ ]:
kodiert = kodiere_beispiel(train[0])
sichtbare_labels = [token for token in kodiert["labels"] if token != -100]

assert len(kodiert["input_ids"]) == len(kodiert["labels"])
assert kodiert["labels"][0] == -100
assert sichtbare_labels[-1] == tokenizer.eos_token_id
assert tokenizer.decode(sichtbare_labels).startswith('{"severity"')

print("✅ Maskierung korrekt")
print("Gesamtlänge:       ", len(kodiert["labels"]), "Tokens")
print("Tokens mit Loss:   ", len(sichtbare_labels))
print("Gelerntes Ziel:    ", tokenizer.decode(sichtbare_labels))


<details>
<summary>💡 Lösung zu Aufgabe 2</summary>

```python
MAX_LAENGE = 256


def kodiere_beispiel(beispiel):
    prompt = baue_prompt(beispiel["beschreibung"])
    ziel = baue_ziel(beispiel["ziel"]) + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    ziel_ids = tokenizer(ziel, add_special_tokens=False)["input_ids"]

    input_ids = (prompt_ids + ziel_ids)[:MAX_LAENGE]
    labels = ([-100] * len(prompt_ids) + ziel_ids)[:MAX_LAENGE]
    return {"input_ids": input_ids, "labels": labels}
```

</details>


## 6 · Adapter trainieren

Jetzt kommt die bekannte Trainingsschleife:

1. `forward`: Vorhersage und Loss berechnen,
2. `backward`: Gradienten berechnen,
3. `step`: **nur die LoRA-Parameter** aktualisieren,
4. `zero_grad`: Gradienten für den nächsten Schritt leeren.

Der Batch-Code ist Infrastruktur und deshalb vorgegeben. Beobachte beim Lauf,
ob der Loss im Mittel sinkt.


In [ ]:
BATCH_GROESSE = 8
SCHRITTE = 400
LERNRATE = 3e-4

trainingsdaten = [kodiere_beispiel(b) for b in train]


def zu_batch(beispiele):
    laenge = max(len(b["input_ids"]) for b in beispiele)
    pad = tokenizer.pad_token_id
    return {
        "input_ids": torch.tensor([
            b["input_ids"] + [pad] * (laenge - len(b["input_ids"])) for b in beispiele
        ]),
        "attention_mask": torch.tensor([
            [1] * len(b["input_ids"]) + [0] * (laenge - len(b["input_ids"]))
            for b in beispiele
        ]),
        "labels": torch.tensor([
            b["labels"] + [-100] * (laenge - len(b["labels"])) for b in beispiele
        ]),
    }


lader = DataLoader(trainingsdaten, batch_size=BATCH_GROESSE, shuffle=True,
                   collate_fn=zu_batch, drop_last=True)
optimizer = torch.optim.AdamW(
    [p for p in modell.parameters() if p.requires_grad], lr=LERNRATE
)

modell.train()
loss_verlauf = []
start = time.time()

while len(loss_verlauf) < SCHRITTE:
    for batch in lader:
        if len(loss_verlauf) >= SCHRITTE:
            break
        batch = {name: tensor.to(GERAET) for name, tensor in batch.items()}
        loss = modell(**batch).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        loss_verlauf.append(loss.item())

        if len(loss_verlauf) % 100 == 0:
            mittel = sum(loss_verlauf[-100:]) / 100
            print(f"Schritt {len(loss_verlauf):3d} · mittlerer Loss {mittel:.3f}")

print(f"Fertig nach {time.time() - start:.0f} Sekunden auf {GERAET}")


In [ ]:
fenster = 25
geglättet = [
    sum(loss_verlauf[max(0, i - fenster):i + 1]) /
    len(loss_verlauf[max(0, i - fenster):i + 1])
    for i in range(len(loss_verlauf))
]

plt.plot(loss_verlauf, color=GRAU, linewidth=0.6, label="Loss je Schritt")
plt.plot(geglättet, color=BLAU, linewidth=2, label="geglättet")
plt.xlabel("Trainingsschritt")
plt.ylabel("Loss")
plt.title("Lernt der Adapter?")
plt.legend()
plt.show()


## 7 · Hat das Training geholfen?

Ein sinkender Trainings-Loss reicht nicht als Beleg. Entscheidend ist die
Leistung auf den **ungesehenen Testfällen**. Wir verwenden exakt dieselbe
Messung wie vorher; nur der Adapter ist neu.


In [ ]:
modell.eval()
nachher = bewerte(modell, test)

print(f"{'Messgröße':<14}{'vorher':>10}{'nachher':>10}")
print("-" * 34)
for name in ["json", *FELDER]:
    print(f"{name:<14}{vorher[name]:>10.0%}{nachher[name]:>10.0%}")


In [ ]:
antwort_nachher = helfer.erzeuge_antworten(
    modell, tokenizer, [prompt_vorher]
)[0]

print("VORHER: ", antwort_vorher)
print("NACHHER:", antwort_nachher)
print("ZIEL:    ", json.dumps(test[0]["ziel"]))


### Ergebnis richtig lesen

- **JSON steigt:** Der Adapter hat das Ausgabeformat gelernt.
- **Feldtreue steigt:** Er hat die Triage-Regeln auf neue Beispiele übertragen.
- **Ein Feld bleibt schwach:** Dafür fehlen möglicherweise Beispiele oder
  Modellkapazität; mehr Trainingsschritte allein sind nicht immer die Lösung.

Die Daten sind synthetisch und folgen festen Regeln. Gute Testwerte zeigen
hier den Lernmechanismus — nicht die Einsatzreife eines echten SOC-Systems.


## 8 · Nur den Adapter speichern

Ein LoRA-Checkpoint enthält die kleinen Zusatzmatrizen und ihre Konfiguration,
nicht das gesamte Basismodell. Zum späteren Laden braucht man daher beides:

```text
Basismodell + gespeicherter Adapter = angepasstes Modell
```


In [ ]:
helfer.AUSGABE.mkdir(exist_ok=True)
adapter_pfad = helfer.AUSGABE / "triage-lora"
modell.save_pretrained(adapter_pfad)

print("Gespeichert in:", adapter_pfad)
print(f"Adaptergröße:    {helfer.groesse_mb(adapter_pfad):.2f} MB")


## Fazit

Du hast den kompletten LoRA-Zyklus durchgeführt:

1. einen passenden Fine-Tuning-Fall erkannt,
2. eine Nullmessung erstellt,
3. weniger als ein Prozent der Parameter trainierbar gemacht,
4. den Prompt vom Loss ausgeschlossen und das Ziel trainiert,
5. auf ungesehenen Daten vorher und nachher verglichen,
6. nur den kleinen Adapter gespeichert.

**Ein Satz zum Mitnehmen:** LoRA verändert nicht das ganze Modell, sondern
lernt eine kleine, austauschbare Korrektur für eine klar definierte Aufgabe.

<details>
<summary>Bonuswissen: QLoRA und Merging</summary>

- **QLoRA** speichert zusätzlich das eingefrorene Basismodell quantisiert,
  meist in 4 Bit. Das spart noch mehr Speicher, benötigt aber passende Hardware
  und Bibliotheken.
- **Merging** rechnet den Adapter in das Basismodell ein. Die Inferenz wird
  einfacher, aber der Vorteil eines kleinen, austauschbaren Artefakts geht
  verloren.

Beides ist für das Grundverständnis nützlich, aber kein notwendiger Teil dieses
ersten Trainingslaufs.

</details>
